# Bayesian networks

_Artificial Intelligence (CS221)_

**Draw arrows showing what causes what. The picture compresses a giant probability table.**

The chance of many things happening together can need a huge table.

     A Bayesian network shrinks it. It is a diagram of arrows: each arrow points from a cause to an effect.

---

This notebook builds a Bayesian network from scratch, one piece at a time. Instead of storing one enormous joint-probability table, we store a small **conditional probability table (CPT)** for each variable and multiply them back together only when we need a probability. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

We'll build the classic **burglary / earthquake / alarm** network. Three boolean variables, but the key idea is that we never write down all eight joint probabilities by hand — we store one small CPT per variable and reconstruct the joint by multiplying. We do it in three steps: (1) the CPTs, (2) the factorized joint, (3) a query and a sanity check.

### Step 1 — Write down the conditional probability tables

Each variable gets its own table. `Burglary` and `Earthquake` have no parents, so each is just a single prior probability. The `Alarm` depends on *both* of them, so its table is indexed by the `(Burglary, Earthquake)` pair — four entries, one per combination of its parents. These tables encode the arrows: B → A and E → A.

In [ ]:
# Variables: Burglary, Earthquake, Alarm.
# P(true) priors for the two root causes (no parents).
pB = {True: 0.001, False: 0.999}
pE = {True: 0.002, False: 0.998}

# P(Alarm=True | Burglary, Earthquake), one row per parent combination.
pA = {
    (True,  True):  0.95,
    (True,  False): 0.94,
    (False, True):  0.29,
    (False, False): 0.001,
}

### Step 2 — Reconstruct the joint by multiplying CPTs

A Bayesian network's central claim is that the full joint probability *factorizes* into a product of the per-variable CPTs: `P(B, E, A) = P(B) · P(E) · P(A | B, E)`. The helper below looks up each factor for a given assignment and multiplies them. When `Alarm` is false we use `1 - P(Alarm=True | ...)`, since the table only stores the probability of the alarm ringing.

In [ ]:
def joint(b, e, a):
    # Look up P(Alarm=a | b, e); the table stores P(True), so flip it when a is False.
    p_alarm = pA[(b, e)] if a else 1 - pA[(b, e)]

    # Factorized joint = product of the three CPT factors.
    return pB[b] * pE[e] * p_alarm

### Step 3 — Query the network and sanity-check it

Now we ask a concrete question: what is the probability of *no burglary, no earthquake, yet the alarm rings*? That is a single entry of the joint, so one call to `joint` answers it. As a correctness check, we also sum the joint over all eight assignments — a valid probability distribution must total exactly 1.

In [ ]:
# Probability of: no burglary, no earthquake, but the alarm rings.
p = joint(False, False, True)
print("P(B=F, E=F, A=T) =", round(p, 8))

# Sanity check: the full joint over all 8 assignments must sum to 1.
total = 0.0
for b in (True, False):
    for e in (True, False):
        for a in (True, False):
            total += joint(b, e, a)
print("sum over all assignments =", round(total, 6))

## Visualize the data & results

_In the classic sprinkler/rain/wet-grass network, how likely is it that the sprinkler ran but it did not rain, given the grass is wet?_

We'll build a second, slightly larger network and answer a marginal query, then chart it. We do it in three steps: (1) the CPTs, (2) the joint plus the marginal sum, (3) the bar chart.

### Step 1 — Build the sprinkler network's CPTs

Here `Cloudy` is the root cause. Both `Sprinkler` and `Rain` depend on it (a cloudy day makes rain more likely and the sprinkler less likely). The grass being `Wet` depends on *both* the sprinkler and the rain, so its table is indexed by the `(Sprinkler, Rain)` pair.

In [ ]:
pC = {True: 0.5, False: 0.5}        # P(Cloudy)
pR = {True: 0.8, False: 0.2}        # P(Rain | Cloudy)
pS = {True: 0.1, False: 0.5}        # P(Sprinkler | Cloudy)

# P(Wet=True | Sprinkler, Rain), one row per parent combination.
pW = {
    (True,  True):  0.99,
    (True,  False): 0.90,
    (False, True):  0.90,
    (False, False): 0.0,
}

### Step 2 — Compute the queried marginal probability

We want `P(Sprinkler=T, Rain=F, Wet=T)`. `Cloudy` is not mentioned in the query, so we **marginalize** it out — sum the joint over both values of `Cloudy`. The joint again factorizes into the product of the four CPTs.

In [ ]:
def joint(c, s, r, w):
    p_sprinkler = pS[c] if s else 1 - pS[c]
    p_rain      = pR[c] if r else 1 - pR[c]
    p_wet       = pW[(s, r)] if w else 1 - pW[(s, r)]
    return pC[c] * p_sprinkler * p_rain * p_wet

# Marginalize out Cloudy: sum the joint over c = True and c = False.
p = sum(joint(c, True, False, True) for c in (True, False))   # S=T, R=F, W=T

vals = [round(p, 4), round(1 - p, 4)]
print("P(S=T, R=F, W=T) =", vals[0])

### Step 3 — Chart the queried probability

Finally we plot the queried event against everything else. The two bars sum to 1, which is a quick visual reminder that we computed a proper probability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

labels = ["P(S=T, R=F, W=T)", "everything else"]
ax.bar(labels, vals, color=["#ffb454", "#4ea1ff"])

# Annotate each bar with its value.
for i, v in enumerate(vals):
    ax.text(i, v, v, ha="center", va="bottom")

ax.set_title("sprinkler network: a queried marginal probability")
plt.show()